# Notebook 08 — How BLAST Actually Works

**Module:** 08 — Bioinformatics: Sequence Analysis  
**Notebook:** 08 of 18  
**Time estimate:** 90 minutes

> **The interview answer.** After this notebook, you must be able to answer
> "How does BLAST work?" with: k-mer seeding → ungapped extension → gapped extension
> → E-value filtering. No notes. Under 90 seconds.

> **Pass-3 target:** Implement `kmer_index()` and `blast_seed_and_extend()` from
> scratch. E-value formula recallable without reference.

---
## Step 1 — Motivation

SW is optimal but O(n×m) — aligning a 300-bp query against the entire GenBank
(3×10¹² bp) would take years. BLAST (Basic Local Alignment Search Tool, 1990)
solves this with heuristics that achieve near-SW sensitivity at ~1000× the speed.
It is the most-used bioinformatics tool in the world, and understanding how it works
— not just that it does — is a direct Track A interview requirement.

---
## Step 2 — Intuition

BLAST's insight: **you don't need to try every possible alignment position**. Instead:

1. **Seed:** find short exact matches (k-mers) between query and database. Only
   positions where a k-mer matches are candidate alignment sites.
2. **Extend:** from each seed, try to extend the alignment in both directions
   (first without gaps for speed, then with gaps for sensitivity).
3. **Filter:** keep only alignments whose score is unlikely to arise by chance.
   This is the E-value.

The k-mer seeding step eliminates ~99.9% of impossible alignment positions instantly,
making the full DP extension affordable only at genuine candidate sites.

---
## Step 3 — Biological Background

BLAST's original paper had two key biological insights:
1. Real homologs share short, exactly-matching subsequences. A random pair rarely shares
   an 11-mer (nucleotide) or even a 3-mer (protein) by chance.
2. The statistical significance of an alignment score can be calculated analytically
   using Extreme Value Distribution (Gumbel distribution) — giving the E-value.

BLAST variants:
| Program | Query | Database |
|---------|-------|----------|
| blastn | nucleotide | nucleotide |
| blastp | protein | protein |
| blastx | translated DNA | protein |
| tblastn | protein | translated DNA |
| tblastx | translated DNA | translated DNA |

---
## Step 4 — Mathematical Explanation

### 4.1 The Four Steps of BLAST

**Step 1: Seed (k-mer lookup)**  
For a query of length $q$, generate all $(q - k + 1)$ k-mers. Look up each in a
pre-built index of the database. For DNA: $k = 11$ (blastn default). For protein:
$k = 3$, but also include k-mers with high BLOSUM62 scores (neighbourhood search).

**Step 2: Ungapped extension**  
From each seed hit at position $(i, j)$ in the query-database pair, extend left and
right without gaps. Score using the substitution matrix. Stop when the score drops
more than $X$ below the running maximum (X-drop). Only High-Scoring Pairs (HSPs)
above threshold $T$ proceed to step 3.

**Step 3: Gapped extension**  
Apply SW with affine gap penalties to the region around the HSP. This is the
computationally expensive step, but it's only done at high-confidence sites.

**Step 4: E-value calculation**  

$$E = K \cdot m \cdot n \cdot e^{-\lambda S}$$

where:
- $S$ = raw alignment score
- $m$ = effective query length
- $n$ = effective database size (total residues)
- $K, \lambda$ = statistical parameters depending on the scoring matrix and gap penalties

**Interpretation:** E is the expected number of alignments with score $\geq S$ that
would arise by chance in a random database of size $n$. $E < 0.001$ = very significant.
$E = 10$ = expect 10 false hits at this score by chance alone.

**Bit score:** $S' = (\lambda S - \ln K) / \ln 2$ — normalized score independent
of database size, enabling comparison across runs.

In [ ]:
# Step 6 — Python Implementation
from __future__ import annotations
import numpy as np
from collections import defaultdict
from dataclasses import dataclass
import math


# Step 1: k-mer index
def kmer_index(sequence: str, k: int) -> dict[str, list[int]]:
    index = defaultdict(list)
    for i in range(len(sequence) - k + 1):
        kmer = sequence[i:i + k]
        index[kmer].append(i)
    return dict(index)


# Step 2: Find seed hits between query and subject
def find_seed_hits(
    query: str,
    subject: str,
    k: int = 11
) -> list[tuple[int, int]]:
    """
    Returns list of (query_pos, subject_pos) for each k-mer hit.
    Each position is the start of the matching k-mer.
    """
    db_index = kmer_index(subject, k)
    hits = []
    for qi in range(len(query) - k + 1):
        qmer = query[qi:qi + k]
        if qmer in db_index:
            for si in db_index[qmer]:
                hits.append((qi, si))
    return hits


# Step 3: Ungapped extension from a seed hit
def ungapped_extension(
    query: str,
    subject: str,
    qi: int,          # query start of seed
    si: int,          # subject start of seed
    k: int,           # seed length (extension starts here)
    match: int = 1,
    mismatch: int = -2,
    x_drop: int = 5   # stop if score drops X below best
) -> tuple[int, int, int, int, int]:
    """
    Extend alignment right from (qi+k, si+k), then left from (qi, si).
    Returns (score, q_start, q_end, s_start, s_end) — 0-indexed, end exclusive.
    """
    def extend_right(q_pos, s_pos):
        score = 0
        best = 0
        best_end = 0
        d = 0
        while q_pos + d < len(query) and s_pos + d < len(subject):
            score += match if query[q_pos + d] == subject[s_pos + d] else mismatch
            if score > best:
                best = score
                best_end = d + 1
            if score < best - x_drop:
                break
            d += 1
        return best, best_end

    def extend_left(q_pos, s_pos):
        score = 0
        best = 0
        best_ext = 0
        d = 1
        while q_pos - d >= 0 and s_pos - d >= 0:
            score += match if query[q_pos - d] == subject[s_pos - d] else mismatch
            if score > best:
                best = score
                best_ext = d
            if score < best - x_drop:
                break
            d += 1
        return best, best_ext

    # Score the seed itself
    seed_score = sum(
        match if query[qi + d] == subject[si + d] else mismatch
        for d in range(k)
    )

    right_score, right_ext = extend_right(qi + k, si + k)
    left_score, left_ext = extend_left(qi, si)

    total_score = seed_score + right_score + left_score
    q_start = qi - left_ext
    q_end = qi + k + right_ext
    s_start = si - left_ext
    s_end = si + k + right_ext

    return total_score, q_start, q_end, s_start, s_end


# E-value calculation
def evalue(
    score: int,
    query_len: int,
    db_size: int,
    lambda_param: float = 1.28,  # for DNA, match=1, mismatch=-2
    K_param: float = 0.46
) -> float:
    return K_param * query_len * db_size * math.exp(-lambda_param * score)


def bit_score(score: int, lambda_param: float = 1.28, K_param: float = 0.46) -> float:
    return (lambda_param * score - math.log(K_param)) / math.log(2)


# Minimal BLAST-like search
@dataclass
class BLASTHit:
    score: int
    evalue: float
    bit_score: float
    q_start: int
    q_end: int
    s_start: int
    s_end: int
    q_aligned: str
    s_aligned: str


def simple_blast(
    query: str,
    subject: str,
    k: int = 7,
    match: int = 1,
    mismatch: int = -2,
    min_score: int = 10,
    max_evalue: float = 0.01,
    db_size: int = None
) -> list[BLASTHit]:
    if db_size is None:
        db_size = len(subject)

    hits = find_seed_hits(query, subject, k)
    results = []
    seen_positions = set()

    for qi, si in hits:
        # Dedup nearby seeds
        key = (qi // k, si // k)
        if key in seen_positions:
            continue
        seen_positions.add(key)

        score, q_start, q_end, s_start, s_end = ungapped_extension(
            query, subject, qi, si, k, match, mismatch
        )

        if score < min_score:
            continue

        ev = evalue(score, len(query), db_size)
        if ev > max_evalue:
            continue

        bs = bit_score(score)
        q_aln = query[q_start:q_end]
        s_aln = subject[s_start:s_end]

        results.append(BLASTHit(
            score=score, evalue=ev, bit_score=bs,
            q_start=q_start, q_end=q_end,
            s_start=s_start, s_end=s_end,
            q_aligned=q_aln, s_aligned=s_aln
        ))

    results.sort(key=lambda x: x.evalue)
    return results


print("k-mer index:")
idx = kmer_index('ATCGATCGATCG', k=4)
print({k: v for k, v in list(idx.items())[:5]})

print()
print("Seed hits:")
query = 'ATCGATCG'
subject = 'GGGATCGATCGCCC'
hits = find_seed_hits(query, subject, k=4)
print(f"Hits (qi, si): {hits}")

In [ ]:
# Demonstrate BLAST search
# Embed a known query region inside a longer subject with flanking noise
rng = np.random.default_rng(42)
bases = list('ACGT')

query = 'ATCGATCGATCGATCG'  # 16 bp query
# Subject: random flanks with the query embedded in the middle
flank = ''.join(rng.choice(bases, 50))
subject = flank + query + ''.join(rng.choice(bases, 50))

print(f"Query: {query}")
print(f"Subject (100 bp with embedded query at pos 50–66):")
print(f"  {subject}")
print()

blast_hits = simple_blast(query, subject, k=7, min_score=8, max_evalue=0.1)
print(f"Found {len(blast_hits)} hit(s):")
for hit in blast_hits:
    print(f"  Score={hit.score}, E={hit.evalue:.4f}, bits={hit.bit_score:.1f}")
    print(f"  Query  [{hit.q_start}:{hit.q_end}]: {hit.q_aligned}")
    print(f"  Subject[{hit.s_start}:{hit.s_end}]: {hit.s_aligned}")
    mid = ''.join('|' if a==b else '.' for a,b in zip(hit.q_aligned, hit.s_aligned))
    print(f"  Match:  {mid}")

In [ ]:
# E-value behavior
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel A: E-value vs. score
ax = axes[0]
scores = np.arange(1, 80)
ev_100_1e6 = [evalue(s, 100, 1_000_000) for s in scores]
ev_100_1e9 = [evalue(s, 100, 1_000_000_000) for s in scores]
ax.semilogy(scores, ev_100_1e6, 'b-', label='DB=1M bp')
ax.semilogy(scores, ev_100_1e9, 'r-', label='DB=1B bp (GenBank)')
ax.axhline(0.001, ls='--', color='green', label='E=0.001 threshold')
ax.axhline(10, ls='--', color='orange', label='E=10 (BLAST default)')
ax.set_xlabel('Alignment score')
ax.set_ylabel('E-value')
ax.set_title('E-value vs. score\n(query length=100 bp)')
ax.legend(fontsize=8)

# Panel B: E-value vs. database size (fixed score)
ax2 = axes[1]
db_sizes = np.logspace(4, 12, 50)
for score_val, label in [(20, 'score=20'), (40, 'score=40'), (60, 'score=60')]:
    evs = [evalue(score_val, 100, int(db)) for db in db_sizes]
    ax2.loglog(db_sizes, evs, label=label)
ax2.axhline(0.001, ls='--', color='green', alpha=0.5)
ax2.set_xlabel('Database size (bp)')
ax2.set_ylabel('E-value')
ax2.set_title('E-value scales linearly with DB size\n(why DB size matters for E-value)')
ax2.legend(fontsize=8)

# Panel C: BLAST speed advantage
ax3 = axes[2]
db_lengths = [100, 500, 1000, 5000, 10000]
query_len = 50

# SW: O(n*m) for every position
sw_ops = [query_len * db for db in db_lengths]

# BLAST: only at k-mer hit positions
# Expected k-mer hits by chance: ~db_length * (1/4^k) for DNA k-mer
k = 7
expected_hits = [db / (4**k) * query_len for db in db_lengths]
blast_ops_per_hit = query_len * query_len  # SW only at hit sites
blast_total = [h * blast_ops_per_hit + db for h, db in zip(expected_hits, db_lengths)]

ax3.plot(db_lengths, [s/1000 for s in sw_ops], 'r-o', label='SW (all positions)')
ax3.plot(db_lengths, [b/1000 for b in blast_total], 'b-s', label='BLAST (only at seeds)')
ax3.set_xlabel('Database length (bp)')
ax3.set_ylabel('Operations (thousands)')
ax3.set_title(f'BLAST vs. SW operation count\n(query={query_len} bp, k={k})')
ax3.legend(fontsize=8)

plt.tight_layout()
plt.savefig('blast_mechanics.png', dpi=150, bbox_inches='tight')
plt.show()

---
## The Interview Answer (write this from memory)

**"How does BLAST work?"**

BLAST finds locally similar regions between a query and a sequence database in three
main stages:

1. **k-mer seeding:** The query is broken into all overlapping k-mers (11 for DNA,
   3 for protein). A pre-built index of the database is searched for exact matches
   to each k-mer. Only positions where a k-mer matches are investigated further.

2. **Extension:** From each seed, the alignment is extended in both directions without
   gaps (fast) until the score drops below a threshold. Promising hits are then
   extended again with affine gap penalties (Smith-Waterman-like), which gives the
   final High-Scoring Pair (HSP).

3. **E-value:** Each HSP's score is converted to an E-value: the expected number of
   alignments with that score or better in a random database of the same size. The
   formula is E = K·m·n·e^(−λS). E < 0.001 is typically considered significant.

BLAST is fast because k-mer seeding eliminates ~99.9% of impossible alignment
positions before any DP is attempted.

---
## Step 8 — Exercises

See `exercises/08_blast_exercises.md`.

1. Calculate the E-value for a BLAST hit with score 50, query length 100,
   database size 10^9. Use K=0.46, λ=1.28. Is this significant?
2. Why does BLAST use k=11 for nucleotide BLAST? What happens if k=3? If k=20?
3. Run blastn on NCBI (ncbi.nlm.nih.gov/blast/) for the human BRCA1 gene
   (first 100 bp). Find the E-value of the top hit and the worst hit returned.
4. Implement `kmer_index()` from memory. What data structure does it use and why?

---
## Step 10 — Quiz

1. What are the four steps of BLAST, in order?
2. What does E-value mean? Interpret E=0.0001 and E=100.
3. What is X-drop in ungapped extension?
4. Why does BLAST use k-mers instead of doing SW at every position?
5. What is a bit score, and why is it useful compared to raw score?

---
## Step 12 — Reflection

> **Practice the interview answer:** Explain BLAST to an imaginary interviewer in
> under 90 seconds. Time yourself. You must be able to say: k-mer seeding,
> ungapped extension, gapped extension, E-value — in that order, correctly.

---
## Step 13 — Papers Referenced

- Altschul et al. (1990) "Basic local alignment search tool." *J Mol Biol* 215(3).
  The original BLAST paper. Pass-2 target.
- Altschul et al. (1997) "Gapped BLAST and PSI-BLAST." *Nucleic Acids Res* 25(17).
  The gapped extension addition. Read abstract and methods.

---
*Next: `09_multiple_sequence_alignment.ipynb`*